# SFSC 2026 — frozen V4 inference benchmark (Colab / L4)

**Inference only. No training, no scorer fitting, no Kaggle submission.** Measures raw 2048² JPEG → flattening → four UNets × D4 → four YOLO11m-seg detectors → frozen V4 fusion/scorer → compressed COCO-RLE.

Select **Python 3 / NVIDIA L4**. Stage this repository at `MyDrive/SFSC2026/repo` and the completed original V3/V4 runs under `MyDrive/SFSC2026/outputs/`. You need all eight trusted checkpoints, each fold's `done.json`, V3 `fold0/config.json` and frozen recipe, plus V4 manifest/scorer. The repo does not bundle these assets. Never load untrusted pickle/checkpoint files.

Budget **5–15 minutes after dependencies/assets are staged** for five images × two repeats (estimate, not measured). Start with one image × one repeat if needed. The benchmark has not been run on a GPU by its author. Model loading/staging is reported separately; dependency installation, result writes and Drive upload are excluded from warm latency. Stop/delete the runtime after saving results.

This temporary adapter imports selected function definitions and constant prefixes from the original notebooks using Python AST. It does **not** execute their top-level training, map-cache replay, evaluation, scorer-fit or submission cells. After the lead extracts shared modules, replace the adapter imports and verify equivalent frozen outputs.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/SFSC2026/repo')  # edit if needed
assert (REPO / 'requirements-colab-cuda.txt').is_file(), REPO

In [ ]:
# Run this cell once, restart the session, then run all OTHER cells.
# The requirements preserve the observed Torch 2.11.0+cu130 build.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r',
                       str(REPO / 'requirements-colab-cuda.txt')])
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)

## Configure paths and sample size

`IMAGE_DIR` must contain only competition **test input images**, not labels. Models/scorer remain frozen. The default selects five evenly spaced sorted filenames, so examples are not chosen by prediction quality. `VERIFY_CACHED=True` checks the warm-up image against available frozen maps/detections outside the timed region; missing optional caches are reported. Per-image latency includes local JPEG IO, flattening and CPU RLE serialization. It excludes CSV writes and network/Drive transfers.

In [ ]:
PROJECT = Path('/content/drive/MyDrive/SFSC2026')
V3_RUN = PROJECT / 'outputs/sfsc_unet_v3'
V4_RUN = PROJECT / 'outputs/sfsc_fusion_v4'
IMAGE_DIR = PROJECT / 'data/SFSCDataset/test'  # recursively discovers JPEGs
LOCAL_ROOT = Path('/content/sfsc_inference_benchmark')
OUTPUT_DIR = PROJECT / 'outputs/benchmark'  # private until reviewed for publication
V3_NOTEBOOK = REPO / 'v3/SFSC2026_V3_UNet_EffV2_4fold_L4_Colab.ipynb'
V4_NOTEBOOK = REPO / 'v4/SFSC2026_V4_Fusion_UNet_YOLO11_L4_Colab (1).ipynb'
N_IMAGES, REPEATS = 5, 2
VERIFY_CACHED = True
assert N_IMAGES > 0 and REPEATS > 0
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import ast, contextlib, gc, hashlib, importlib.metadata, json, math, os
import pickle, platform, resource, shutil, subprocess, sys, time, types
from collections import defaultdict
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from pycocotools import mask as mask_api
import torch
from torch import nn
from torch.nn import functional as F
import segmentation_models_pytorch as smp
from ultralytics import YOLO

assert torch.cuda.is_available(), 'Select an NVIDIA L4 GPU runtime.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'This benchmark protocol specifies L4, found {GPU}'
assert torch.__version__ == '2.11.0+cu130', torch.__version__
for package, version in {'torchvision': '0.26.0+cu130', 'ultralytics': '8.4.175',
                         'segmentation-models-pytorch': '0.5.0',
                         'timm': '1.0.30', 'scikit-learn': '1.6.1'}.items():
    actual = importlib.metadata.version(package)
    assert actual == version, f'{package}: expected {version}, found {actual}'
cv2.setNumThreads(1)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text())

v3_config = read_json(V3_RUN / 'fold0/config.json')['config']
v4_manifest = read_json(V4_RUN / 'manifest_v4.json')
frozen = read_json(V3_RUN / 'frozen_recipe_v3.json')
assert v4_manifest['v3_recipe_sha256'] == frozen['sha256'], 'Mixed V3/V4 recipe artifacts'
assert v4_manifest['adopted'] == 'v4_full', 'Expected the shipped V4 variant'
with (V4_RUN / 'scorer_v4.pkl').open('rb') as f:
    scorer = pickle.load(f)  # own trusted artifact only
assert scorer['adopted'] == v4_manifest['adopted']
recipe_geometry = frozen['recipe']['geometry']
geom = (float(recipe_geometry['t_low']), recipe_geometry['contact'], int(recipe_geometry['offset']))
assert tuple(scorer['geometry']) == geom == (0.4, 0.5, -1)
folds = sorted(map(int, frozen['recipe']['folds']))
assert folds == [0, 1, 2, 3]
assert sorted(map(int, v4_manifest['detector']['folds'])) == folds
assert float(scorer['final']['v4_full']['tau']) == 0.225
assert all((V3_RUN / f'fold{f}/done.json').is_file() and
           (V4_RUN / f'detector/fold{f}/done.json').is_file() for f in folds)

## Load the original inference definitions and frozen weights

The helper rejects missing/ambiguous definitions and copies checkpoints/images to the local SSD before timing. It records hashes and verifies checkpoints against `done.json`. Four UNets and four detectors remain resident during the test; peak CUDA memory includes that residency. The only top-level notebook statements imported are explicitly identified constant prefixes; neural training and HGB fitting are never imported or called.

In [ ]:
def read_notebook(path):
    notebook = read_json(path)
    return [ast.parse(''.join(c['source'])) for c in notebook['cells'] if c['cell_type'] == 'code']

def definition(trees, name):
    found = [node for tree in trees for node in tree.body
             if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name]
    assert len(found) == 1, f'Expected one definition of {name}, found {len(found)}'
    return found[0]

def exec_nodes(nodes, namespace):
    module = ast.fix_missing_locations(ast.Module(body=nodes, type_ignores=[]))
    exec(compile(module, '<selected notebook inference>', 'exec'), namespace)

def prefix_before(trees, function_name, namespace):
    matches = [tree for tree in trees if any(isinstance(n, ast.FunctionDef) and
                                             n.name == function_name for n in tree.body)]
    assert len(matches) == 1
    prefix = []
    for node in matches[0].body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            break
        prefix.append(node)
    # Prefixes used below contain only assignments and deletion of temporary arrays.
    assert all(isinstance(n, (ast.Assign, ast.AnnAssign, ast.Delete)) for n in prefix)
    exec_nodes(prefix, namespace)

v3_trees, v4_trees = read_notebook(V3_NOTEBOOK), read_notebook(V4_NOTEBOOK)
module = types.ModuleType('sfsc_benchmark_adapter')
sys.modules[module.__name__] = module
ns = module.__dict__
ns.update({key: value for key, value in globals().copy().items()
           if key in ('Path', 'contextlib', 'math', 'cv2', 'np', 'pd', 'Image', 'ndi',
                      'linear_sum_assignment', 'mask_api', 'torch', 'nn', 'F', 'smp')})
ns['CFG'] = types.SimpleNamespace(**{**v3_config, **v4_manifest['config']})
ns.update(H=2048, W=2048, DEVICE=torch.device('cuda'), AMP_ENABLED=bool(v3_config['amp']),
          AMP_DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
          RUN=LOCAL_ROOT / 'unet', GEOM=geom)
assert tuple(v3_config['expected_shape']) == (2048, 2048)
assert not v3_config['debug_run'] and not v4_manifest['config']['debug_run']
assert all(read_json(V3_RUN / f'fold{fold}/done.json')['encoder'] ==
           'tu-tf_efficientnetv2_s.in21k_ft_in1k' for fold in folds)
assert v3_config['architecture'] == 'Unet'
assert v3_config['tile'] == 1024 and v3_config['stride'] == 768
assert v3_config['infer_batch'] == 4
assert len(v3_config['tta_final']) == 8
assert v4_manifest['detector']['weights'] == 'yolo11m-seg.pt'
assert v4_manifest['detector']['imgsz'] == 1280
prefix_before(v3_trees, 'flatten_disk', ns)  # disk geometry, normalization constants
prefix_before(v3_trees, 'tta_forward', ns)   # D4 operation map and radial GPU cache
prefix_before(v4_trees, 'qt', ns)            # connectivity, component feature names
prefix_before(v4_trees, 'rasterize_detections', ns)  # detector feature names and grow kernel
v3_functions = ['autocast_context', 'flatten_disk', 'build_model', 'to_device',
                'tta_forward', 'tta_inverse', 'tile_starts', 'make_input', 'predict_maps', 'load_fold_model']
v4_functions = ['qt', 'relabel', 'fill_label_holes', 'geometry_labels', 'component_features',
                'prefilter_keep', 'rasterize_detections', 'detection_evidence', 'largest_piece',
                'consensus_scores', 'detector_only_candidates', 'build_candidates', 'detect']
exec_nodes([definition(v3_trees, name) for name in v3_functions], ns)
exec_nodes([definition(v4_trees, name) for name in v4_functions], ns)
assert tuple(scorer['features']) == ns['ALL_FEATURES'], 'Scorer feature order mismatch'
assert ns['N_V3F'] == 12 and ns['N_DETF'] == 3
assert ns['CFG'].det_imgsz == 1280 and ns['CFG'].det_conf_floor == 0.05
assert ns['CFG'].det_nms_iou == 0.6 and ns['CFG'].det_max_det == 200

preparation_started = time.perf_counter()
artifact_hashes = {'v3_notebook': sha256(V3_NOTEBOOK), 'v4_notebook': sha256(V4_NOTEBOOK),
                   'scorer': sha256(V4_RUN / 'scorer_v4.pkl'),
                   'v3_recipe_file': sha256(V3_RUN / 'frozen_recipe_v3.json'),
                   'v3_config': sha256(V3_RUN / 'fold0/config.json'),
                   'v4_manifest': sha256(V4_RUN / 'manifest_v4.json')}
for fold in folds:
    for kind, source, expected in [
        ('unet', V3_RUN / f'fold{fold}/best.pt', read_json(V3_RUN / f'fold{fold}/done.json')['best_sha256']),
        ('detector', V4_RUN / f'detector/fold{fold}/best.pt',
         read_json(V4_RUN / f'detector/fold{fold}/done.json')['weights_sha256'])]:
        target = LOCAL_ROOT / kind / f'fold{fold}/best.pt'
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, target)
        actual = sha256(target)
        assert actual == expected, f'Checkpoint hash mismatch: {kind} fold {fold}'
        artifact_hashes[f'{kind}_fold{fold}'] = actual
checkpoint_staging_s = time.perf_counter() - preparation_started
load_started = time.perf_counter()
models = [ns['load_fold_model'](fold) for fold in folds]
detectors = [YOLO(str(LOCAL_ROOT / f'detector/fold{fold}/best.pt')) for fold in folds]
for detector in detectors:
    detector.to('cuda')
torch.cuda.synchronize()
model_loading_s = time.perf_counter() - load_started
print(f'GPU: {GPU}; checkpoint staging {checkpoint_staging_s:.1f}s; model loading {model_loading_s:.1f}s')

In [ ]:
paths = sorted(p for p in IMAGE_DIR.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg'))
assert len(paths) == 180, f'Expected 180 competition test JPEGs; found {len(paths)}. Fix IMAGE_DIR.'
assert len({p.name for p in paths}) == len(paths), 'Duplicate image filenames'
indices = np.linspace(0, len(paths) - 1, min(N_IMAGES, len(paths)), dtype=int)
sample_paths = []
for index in indices:
    source = paths[int(index)]
    target = LOCAL_ROOT / 'images' / source.name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, target)
    sample_paths.append(target)
image_hashes = {path.name: sha256(path) for path in sample_paths}
final_scorer = scorer['final'][scorer['adopted']]

def synchronized_time():
    torch.cuda.synchronize()
    return time.perf_counter()

def infer_one(path):
    torch.cuda.reset_peak_memory_stats()
    t0 = synchronized_time()
    with Image.open(path) as source:
        img = np.asarray(source.convert('L')).copy()
    assert img.shape == (2048, 2048)
    flat = ns['flatten_disk'](img)
    t1 = synchronized_time()
    fg, ct = ns['predict_maps'](models, img, flat, ns['CFG'].tta_final)
    t2 = synchronized_time()
    detections = [ns['detect'](detector, path) for detector in detectors]
    t3 = synchronized_time()
    lab, n, n_v3, features, areas = ns['build_candidates'](fg, ct, img, flat, detections)
    scores = np.full(n, -np.inf)
    selected_source = np.ones(n, bool) if final_scorer['all'] else features[:, -1] == 0
    if selected_source.any():
        start, stop = final_scorer['cols']
        scores[selected_source] = final_scorer['model'].predict(features[selected_source, start:stop])
    keep_ids = np.flatnonzero(scores >= final_scorer['tau']) + 1
    t4 = synchronized_time()
    rles = [mask_api.encode(np.asfortranarray((lab == label).astype(np.uint8))) for label in keep_ids]
    rows = [{'filament_id': f'{path.stem}_{i}', 'segmentation_rle': rle['counts'].decode('ascii')}
            for i, rle in enumerate(rles, 1)]
    t5 = synchronized_time()
    # Correctness checks are outside the latency region.
    seen = np.zeros(img.shape, bool)
    for rle, label in zip(rles, keep_ids):
        mask = mask_api.decode(rle).astype(bool)
        assert mask.any() and not np.any(seen & mask)
        assert np.array_equal(mask, lab == label)
        seen |= mask
    result = {'file_name': path.name, 'preprocess_s': t1-t0, 'unet_s': t2-t1,
              'detector_s': t3-t2, 'fusion_s': t4-t3, 'rle_s': t5-t4, 'total_s': t5-t0,
              'peak_allocated_gib': torch.cuda.max_memory_allocated() / 2**30,
              'peak_reserved_gib': torch.cuda.max_memory_reserved() / 2**30,
              'process_peak_rss_gib': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20,
              'candidates': n, 'unet_candidates': n_v3, 'instances': len(rows)}
    return result, rows, (fg, ct), detections

warmup, _, warm_maps, warm_detections = infer_one(sample_paths[0])
print('Warm-up elapsed (excluded):', round(warmup['total_s'], 3), 's')
cache_checks = {'enabled': VERIFY_CACHED}
if VERIFY_CACHED:
    cached = V3_RUN / 'test_probs' / f'{sample_paths[0].stem}_{frozen["sha256"][:8]}.npz'
    if cached.is_file():
        with np.load(cached, allow_pickle=False) as stored:
            differences = {key: int(np.abs(actual.astype(np.int16) - stored[key].astype(np.int16)).max())
                           for key, actual in zip(('fg', 'ct'), warm_maps)}
        cache_checks['dense_max_abs_u8'] = differences
        assert max(differences.values()) <= 1, f'Dense-map drift: {differences}'
    else:
        cache_checks['dense_maps'] = 'optional cache absent'
    for fold, actual in zip(folds, warm_detections):
        suffix = artifact_hashes[f'detector_fold{fold}'][:8]
        cached = V4_RUN / 'detections' / f'test_fold{fold}_{suffix}.pkl'
        if not cached.is_file():
            cache_checks[f'detector_fold{fold}'] = 'optional cache absent'
            continue
        with cached.open('rb') as f:
            stored = pickle.load(f)[sample_paths[0].name]  # own trusted artifact only
        assert np.allclose(actual['scores'], stored['scores'], atol=1e-4, rtol=1e-4)
        assert len(actual['polys']) == len(stored['polys'])
        assert all(np.allclose(a, b, atol=0.5, rtol=1e-4)
                   for a, b in zip(actual['polys'], stored['polys']))
        cache_checks[f'detector_fold{fold}'] = 'agrees within tolerance'
print('Cache checks:', cache_checks)
del warm_maps, warm_detections
gc.collect()  # keep model/allocator warm; do not empty the CUDA cache between timed images

In [ ]:
records, expected_rows = [], {}
for repeat in range(REPEATS):
    for path in sample_paths:
        record, rows, _, _ = infer_one(path)
        if path.name in expected_rows:
            assert rows == expected_rows[path.name], f'Non-deterministic RLE output: {path.name}'
        else:
            expected_rows[path.name] = rows
        record['repeat'] = repeat
        records.append(record)
frame = pd.DataFrame(records)
metrics = ('preprocess_s', 'unet_s', 'detector_s', 'fusion_s', 'rle_s', 'total_s')
summary = {name: {'median': float(frame[name].median()), 'p90': float(frame[name].quantile(.9)),
                  'mean': float(frame[name].mean())} for name in metrics}
summary.update(images=len(sample_paths), repeats=REPEATS, timed_samples=len(frame),
               images_per_second=float(len(frame) / frame.total_s.sum()),
               peak_allocated_gib=float(frame.peak_allocated_gib.max()),
               peak_reserved_gib=float(frame.peak_reserved_gib.max()),
               process_peak_rss_gib=float(frame.process_peak_rss_gib.max()),
               checkpoint_staging_s=checkpoint_staging_s, model_loading_s=model_loading_s,
               warmup_total_s=warmup['total_s'])
packages = {name: importlib.metadata.version(name) for name in
            ('torch', 'torchvision', 'segmentation-models-pytorch', 'timm', 'ultralytics',
             'scikit-learn', 'numpy', 'scipy', 'pandas', 'Pillow', 'opencv-python',
             'pycocotools', 'matplotlib', 'tqdm', 'packaging')}
gpu_runtime = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
                              '--format=csv,noheader'], capture_output=True, text=True, check=True).stdout.strip()
metadata = {'protocol': 'V4 raw JPEG to RLE; 4 dense folds x D4, 4 detectors; local SSD; batch=1 image',
            'measured_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            'python': platform.python_version(), 'gpu': GPU, 'gpu_runtime': gpu_runtime,
            'cuda': torch.version.cuda, 'packages': packages,
            'colab_release_tag': os.environ.get('COLAB_RELEASE_TAG'),
            'colab_backend_version': os.environ.get('COLAB_BACKEND_VERSION'),
            'artifact_sha256': artifact_hashes, 'image_sha256': image_hashes,
            'v3_config': v3_config, 'v4_config': v4_manifest['config'],
            'geometry': geom, 'variant': scorer['adopted'], 'tau': final_scorer['tau'],
            'cache_checks': cache_checks,
            'latency_excludes': ['dependency install', 'asset staging', 'model loading', 'warmup',
                                 'correctness checks', 'CSV write', 'Drive/network IO'],
            'memory_scope': 'CUDA allocator peaks include resident models; reserved may reflect warmup; RSS is process lifetime peak'}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
frame.to_csv(OUTPUT_DIR / 'inference_per_image.csv', index=False)
(OUTPUT_DIR / 'inference_summary.json').write_text(json.dumps({'summary': summary, 'metadata': metadata}, indent=2))
print(json.dumps(summary, indent=2))
print('Saved benchmark outputs to', OUTPUT_DIR)
print('Download results, then disconnect/delete this runtime now.')

## Interpretation and reporting

Publish the measured `total_s` median/p90, allocator allocated/reserved peak, GPU/runtime and exact configuration together. Include `inference_per_image.csv` and a sanitized `inference_summary.json` with the report. Five selected images are a smoke benchmark, not a dataset-wide throughput guarantee; report the sample count. Model residency and warm-up allocations are included in the reported CUDA peak; driver/runtime memory outside the PyTorch allocator is not.

Per-image timing includes RLE encoding and Python submission-row construction but excludes full CSV/Drive writes and audit checks. The notebook makes no claim about training cost. Missing optional cache comparisons must be disclosed; a passing benchmark without reference caches measures the frozen recipe but has not independently checked agreement with the historical maps/detections.

Do not rerun training or reduce ensemble/TTA settings to fix drift or memory errors. Inspect the runtime/dependency/artifact mismatch first. Save measured compute-unit usage from the Colab UI if billing evidence is needed; this notebook does not query the account or estimate remaining balance.